# Multimodal Ceramic Glaze Retrieval Engine — Notebook workflow

Run cells in order. Enter an **image file path**. Models are loaded lazily and use the existing Hugging Face cache.

### Install dependencies (once per environment)
Use a Python 3.11 kernel. Run `%pip install torch torchvision` and `%pip install -r requirements-notebook.txt` in a new code cell, then restart the kernel. For NVIDIA CUDA, install the correct PyTorch build first. See `README.md`.

In [ ]:
!pip install torch torchvision

In [ ]:
!pip install -r requirements-notebook.txt

## 1 · Setup
Open this notebook from the project directory. Install dependencies in your notebook kernel environment if needed (see `NOTEBOOK_SETUP.md`).

In [ ]:
from pathlib import Path
import importlib.util, json, time, html
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display, HTML, Markdown
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "retrieval_service.py").exists():
    raise FileNotFoundError(f"Open Jupyter from the project folder containing retrieval_service.py: {PROJECT_ROOT}")
CONTENT = json.loads((PROJECT_ROOT / "demo_content.json").read_text(encoding="utf-8"))
print("Project:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available(), "MPS:", torch.backends.mps.is_available())
print("Segmentation device:", "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

## 2 · Choose an image
Set `IMAGE_PATH` to a local JPG, PNG or WebP file. Relative paths are resolved from the project folder. You can also use a file from `sample_images/`.

In [ ]:
IMAGE_PATH = "sample_images/light_blue.jpg"  # <-- change this to your image path
path = Path(IMAGE_PATH).expanduser()
if not path.is_absolute():
    path = PROJECT_ROOT / path
if not path.is_file():
    raise FileNotFoundError(f"Image not found: {path}")
photo = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
print("Image:", path.name, "·", photo.size)
display(photo.copy().resize((min(450, photo.width), round(photo.height * min(450, photo.width)/photo.width))))

## 3 · Detect and segment ceramic objects
First execution may download Grounding DINO, SAM 2.1 and OpenCLIP weights. Subsequent executions reuse the cached models.

In [ ]:
from segmentation_frontend import load_segmenter, extract_objects
if "segmenter" not in globals():
    print("Loading segmentation models...")
    start_load = time.perf_counter()
    segmenter = load_segmenter()
    print(f"Models loaded in {time.perf_counter()-start_load:.1f}s")
start = time.perf_counter()
objects, segmentation_metadata = extract_objects(photo, segmenter)
segmentation_seconds = time.perf_counter() - start
print(f"Detected {len(objects)} object(s) in {segmentation_seconds:.1f}s")
if not objects:
    print("No ceramic objects detected. Try a clearer photo.")
else:
    for i, obj in enumerate(objects):
        print(f"Object {i}: area={obj['area']:.1%}, primary={obj['primary']}, bbox={obj['box']}")
        display(obj['image'].copy().thumbnail((100,100)) if False else obj['image'].copy())

## 4 · Select an object and extract its glaze colours
Set `OBJECT_INDEX` using the numbered detections above. `COLOR_INDEX` chooses which extracted colour is used for retrieval.

In [ ]:
OBJECT_INDEX = 0  # <-- select detected object (0-based)
if not objects:
    raise RuntimeError("No objects detected; change IMAGE_PATH and rerun cells 2–3.")
if not 0 <= OBJECT_INDEX < len(objects):
    raise IndexError(f"OBJECT_INDEX must be between 0 and {len(objects)-1}")
selected_object = objects[OBJECT_INDEX]
palette_path = PROJECT_ROOT / "scripts" / "17_colorthief_colors.py"
spec = importlib.util.spec_from_file_location("gc2_palette", palette_path)
palette_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(palette_module)
palette, palette_note = palette_module.colorthief_palette(selected_object["image"])
print(palette_note or "Extracted colours:")
for i, item in enumerate(palette):
    display(HTML(f'<span style="display:inline-block;width:25px;height:20px;background:{html.escape(item["hex"])};border:1px solid #aaa;vertical-align:middle"></span> {i}: {html.escape(item["name"])} — {html.escape(item["hex"])}'))
if not palette:
    raise RuntimeError("No colours were extracted from the selected object.")
COLOR_INDEX = 0  # <-- choose colour (0-based)
if not 0 <= COLOR_INDEX < len(palette):
    raise IndexError("COLOR_INDEX is outside the palette range")
selected_color = palette[COLOR_INDEX]
print("Selected:", selected_color)

## 5 · Retrieve regular and diversified results
Both result sets are computed **once**. Changing the display below does not rerun retrieval.

K-means clustering was performed to group recipes based on their oxide composition, but cluster results are not used in the retrieval process. To explore other chemical groups, 'Diversify' mode can be enabled to explore alternative chemical formulations. Instead of returning only the highest-scoring recipes, a maximum of two recipes from each cluster is selected. This allows users to discover glazes with different chemical compositions that may still achieve similar appearances. Although some diversified results may have lower similarity scores, they offer a wider variety of potential glaze formulations to explore.

In [ ]:
SURFACE = "Glossy"    # e.g. Glossy, Matte, Satin
CONE = "6"           # firing cone
NUMBER_OF_RECIPES = 8  # 3 through 8
MAX_PER_CLUSTER = 2
if not 3 <= NUMBER_OF_RECIPES <= 8:
    raise ValueError("NUMBER_OF_RECIPES must be between 3 and 8")
import retrieval_service
params = dict(color=selected_color["name"], surface=SURFACE, cone=CONE,
              query_hex=selected_color["hex"], top_k=NUMBER_OF_RECIPES)
start = time.perf_counter()
regular_results = retrieval_service.retrieve_glazes(**params, diversify=False)
diversified_results = retrieval_service.retrieve_glazes(**params, diversify=True, max_per_cluster=MAX_PER_CLUSTER)
print(f"Retrieved both lists in {time.perf_counter()-start:.1f}s")
print("Regular IDs:", regular_results["recipe_id"].tolist())
print("Diversified IDs:", diversified_results["recipe_id"].tolist())

## 6 · View and compare results
Use the selector to switch between the **already computed** result sets. If `ipywidgets` is unavailable, both sets are displayed below instead.
Recipe images are fetched from the internet.

In [ ]:

# ============================================================
# 6 · View and compare results
# ============================================================

import html
import re
import pandas as pd
import ipywidgets as widgets

from urllib.parse import quote
from IPython.display import display

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SHOW_RECIPE_IMAGES = True

font = CONTENT.get("font", "Arial")
button_color = CONTENT.get("button_color", "#2E6D64")

if not isinstance(button_color, str) or not re.fullmatch(
    r"#[0-9a-fA-F]{6}", button_color
):
    button_color = "#2E6D64"

# Only allow safe CSS font names
ALLOWED_FONTS = {
    "Arial", "Helvetica", "Georgia", "Verdana",
    "Trebuchet MS", "Times New Roman",
    "Courier New", "system-ui"
}

if font not in ALLOWED_FONTS:
    font = "Arial"

recipe_fields = CONTENT.get(
    "recipe_fields",
    [{"field": "recipe_id", "label": "Recipe ID"}]
)

title_field = CONTENT.get(
    "recipe_title_field", "recipe_id"
)

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def value_or_none(value):
    if value is None:
        return None

    try:
        if bool(pd.isna(value)):
            return None
    except (TypeError, ValueError):
        pass

    return str(value).strip() or None


def render_recipe_cards(frame, diversified=False):

    cards = []

    for _, recipe in frame.iterrows():

        recipe_id = value_or_none(
            recipe.get("recipe_id")
        ) or "Unknown"

        # Image
        image_url = value_or_none(
            recipe.get("image_path")
        )

        if (
            SHOW_RECIPE_IMAGES
            and image_url
            and image_url.startswith(("http://", "https://"))
        ):
            image_html = f"""
                <img
                    src="{html.escape(image_url, quote=True)}"
                    loading="lazy"
                    style="
                        width:100%;
                        height:145px;
                        object-fit:contain;
                    "
                >
            """
        else:
            image_html = """
                <div style="color:#888;">
                    No image
                </div>
            """

        # Title
        name = value_or_none(
            recipe.get(title_field)
        ) or f"Recipe {recipe_id}"

        # Recipe metadata
        details = []

        score = recipe.get("final_score")
        try:
            score_number = float(score)
            if pd.notna(score_number):
                details.append(f"""
                    <div style="margin-bottom:5px;">
                        <b>Similarity score:</b> {score_number * 100:.2f}%
                    </div>
                """)
        except (TypeError, ValueError):
            pass

        # All three scores are visible by default in the notebook.
        for field, label in (
            ("latent_score", "Latent score"),
            ("color_score", "Colour score"),
        ):
            try:
                score_value = float(recipe.get(field))
                if pd.notna(score_value):
                    details.append(f"""
                        <div style="margin-bottom:5px;">
                            <b>{label}:</b> {score_value * 100:.2f}%
                        </div>
                    """)
            except (TypeError, ValueError):
                pass

        # Cluster labels are shown only for diversified results.
        if diversified:
            cluster_value = value_or_none(recipe.get("kmeans_cluster"))
            if cluster_value is not None:
                details.append(f"""
                    <div style="margin-bottom:5px;">
                        <b>Cluster:</b> {html.escape(cluster_value)}
                    </div>
                """)

        for item in recipe_fields:

            if not isinstance(item, dict):
                continue

            field = item.get("field")

            if not isinstance(field, str):
                continue

            value = value_or_none(
                recipe.get(field)
            )

            if value is None:
                continue

            label = item.get("label", field)

            details.append(f"""
                <div style="margin-bottom:5px;">
                    <b>{html.escape(str(label))}:</b>
                    {html.escape(value)}
                </div>
            """)

        # Recipe URL
        recipe_url = (
            "https://glazy.org/recipes/"
            + quote(recipe_id, safe="")
        )

        # Card HTML
        cards.append(f"""
            <article style="
                border:1px solid #ddd;
                border-radius:10px;
                padding:12px;
                display:flex;
                flex-direction:column;
                gap:12px;
                min-width:0;
                background:white;
                color:#222;
                box-sizing:border-box;
            ">

                <div style="
                    height:155px;
                    background:#f7f7f7;
                    display:flex;
                    align-items:center;
                    justify-content:center;
                ">
                    {image_html}
                </div>

                <div style="
                    font-weight:700;
                    font-size:15px;
                    min-height:35px;
                    overflow-wrap:anywhere;
                ">
                    {html.escape(name)}
                </div>

                <div style="
                    flex:1;
                    font-size:13px;
                    line-height:1.5;
                    overflow-wrap:anywhere;
                ">
                    {''.join(details)}
                </div>

                <a
                    href="{html.escape(recipe_url, quote=True)}"
                    target="_blank"
                    rel="noopener noreferrer"
                    style="
                        display:block;
                        background:{button_color};
                        color:white;
                        text-align:center;
                        padding:10px;
                        border-radius:7px;
                        text-decoration:none;
                        font-weight:600;
                    "
                >
                    View recipe
                </a>

            </article>
        """)

    # Return an HTML STRING, not an IPython HTML object
    return f"""
        <div style="
            font-family:'{font}',sans-serif;
            display:grid;
            grid-template-columns:
                repeat(auto-fit,minmax(190px,1fr));
            gap:14px;
            width:100%;
            margin-top:12px;
            margin-bottom:20px;
        ">
            {''.join(cards)}
        </div>
    """


# ------------------------------------------------------------
# 3. Prepare both result sets
# ------------------------------------------------------------

# These DataFrames come from Cell 5.
regular_html = render_recipe_cards(regular_results, diversified=False)
diversified_html = render_recipe_cards(diversified_results, diversified=True)

# ------------------------------------------------------------
# 4. Close previous widgets when rerunning
# ------------------------------------------------------------

for name in (
    "results_toggle",
    "results_html",
    "results_container"
):
    old_widget = globals().get(name)

    if isinstance(old_widget, widgets.Widget):
        old_widget.close()

# ------------------------------------------------------------
# 5. Create toggle
# ------------------------------------------------------------

results_toggle = widgets.ToggleButtons(
    options=[
        ("Regular", "regular"),
        ("Diversified", "diversified")
    ],
    value="regular",
    description="Results:",
    style={"description_width": "initial"},
)

# ONE HTML widget displays ONE result set.
results_html = widgets.HTML(
    value=regular_html,
    layout=widgets.Layout(width="100%")
)

# ------------------------------------------------------------
# 6. Update HTML when toggle changes
# ------------------------------------------------------------

def update_results(change):

    selected = change["new"]

    if selected == "regular":
        results_html.value = regular_html

    elif selected == "diversified":
        results_html.value = diversified_html


results_toggle.observe(
    update_results,
    names="value"
)

# ------------------------------------------------------------
# 7. Display only once
# ------------------------------------------------------------

results_container = widgets.VBox(
    [
        results_toggle,
        results_html
    ],
    layout=widgets.Layout(width="100%")
)

display(results_container)


## 7 · Export results
Both CSVs are saved into `notebook_results/` for inspection and comparison.

In [ ]:
output_dir = PROJECT_ROOT / "notebook_results"
output_dir.mkdir(exist_ok=True)
regular_results.to_csv(output_dir / "regular_results.csv", index=False)
diversified_results.to_csv(output_dir / "diversified_results.csv", index=False)
print("Saved:", output_dir / "regular_results.csv")
print("Saved:", output_dir / "diversified_results.csv")